In [8]:
import ast
from pathlib import Path

import pandas as pd


In [9]:
dataset_path = Path('../dataset')

df = pd.read_csv(dataset_path / 'item_metadata.csv')

def parse_list_column(val):
    if isinstance(val, str):
        return ast.literal_eval(val)
    return [] if pd.isna(val) else val

df["ingredients"] = df["ingredients"].apply(parse_list_column)
df["tags"] = df["tags"].apply(parse_list_column)

df.head(2)

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,chicken lickin good pork chops,63986,500,14664,2003-06-06,"[weeknight, time-to-make, course, main-ingredi...","[105.7, 8.0, 0.0, 26.0, 5.0, 4.0, 3.0]",5,"['dredge pork chops in mixture of flour , salt...",here's and old standby i enjoy from time to ti...,"[lean pork chops, flour, salt, dry mustard, ga...",7
1,how i got my family to eat spinach spinach ca...,25775,50,37305,2002-04-22,"[60-minutes-or-less, time-to-make, course, mai...","[166.1, 16.0, 6.0, 32.0, 19.0, 26.0, 3.0]",5,"['preheat oven to 350 degrees', 'place spinach...","if spinach scares you, this is one recipe that...","[frozen chopped spinach, egg, salt, black pepp...",8


In [10]:
df["ingredients"].explode().unique()

<StringArray>
[       'lean pork chops',                  'flour',                   'salt',
            'dry mustard',          'garlic powder',                    'oil',
      'chicken rice soup', 'frozen chopped spinach',                    'egg',
           'black pepper',
 ...
     'coarse ground beef', 'unsalted chicken stock',         'sultana raisin',
     'mexican-style corn',      'whole wheat bagel',              'dry bread',
                   'ribs',               'crackers',        'pecorino cheese',
         'fresh zucchini']
Length: 3869, dtype: str

In [11]:
frecuencia_tags = df["tags"].explode().value_counts()
print(frecuencia_tags.head(10), frecuencia_tags==1)



tags
preparation         7042
time-to-make        6992
course              6752
dietary             6148
main-ingredient     5638
easy                4212
occasion            4154
equipment           3797
cuisine             3134
low-in-something    3111
Name: count, dtype: int64 tags
preparation            False
time-to-make           False
course                 False
dietary                False
main-ingredient        False
                       ...  
saudi-arabian           True
from-scratch            True
nepalese                True
tarts                   True
unprocessed-freezer     True
Name: count, Length: 438, dtype: bool


In [12]:
from collections import Counter
from itertools import chain

ing_counts = Counter(chain.from_iterable(df["ingredients"]))

singles = sum(1 for c in ing_counts.values() if c == 1)
under_five = sum(1 for c in ing_counts.values() if c < 5)

print(f"Total ingredientes: {len(ing_counts)}")
print(f"Aparecen solo 1 vez: {singles} ({singles / len(ing_counts):.1%})")
print(f"Aparecen menos de 5 veces: {under_five} ({under_five / len(ing_counts):.1%})")

Total ingredientes: 3869
Aparecen solo 1 vez: 1548 (40.0%)
Aparecen menos de 5 veces: 2655 (68.6%)


In [14]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer


def comparar_cobertura_min_df(
    series_ingredientes: pd.Series,
    umbrales: list[int] = [5, 10],
    min_ingredientes: int = 3,
) -> pd.DataFrame:
    """Compara el impacto de distintos umbrales min_df sobre la cobertura de ingredientes por receta."""
    total_recetas = len(series_ingredientes)
    reporte = []

    for df_val in umbrales:
        vec = CountVectorizer(
            tokenizer=lambda x: x,
            preprocessor=lambda x: x,
            lowercase=False,
            binary=True,
            min_df=df_val,
        )
        # Matriz dispersa: (N_recetas, vocabulario)
        matriz_sparse = vec.fit_transform(series_ingredientes)

        # Conteo de ingredientes reconocidos por cada receta
        ingredientes_por_receta = np.asarray(matriz_sparse.sum(axis=1)).ravel()

        # Métricas clave
        con_al_menos_k = (ingredientes_por_receta >= min_ingredientes).sum()
        con_cero = (ingredientes_por_receta == 0).sum()

        reporte.append({
            "min_df": df_val,
            "vocabulario": len(vec.vocabulary_),
            f"recetas_>=_{min_ingredientes}": con_al_menos_k,
            f"%_recetas_>=_{min_ingredientes}": f"{(con_al_menos_k / total_recetas):.2%}",
            "recetas_huerfanas_(=0)": con_cero,
            "%_huerfanas": f"{(con_cero / total_recetas):.2%}",
            "promedio_ingredientes": round(float(np.mean(ingredientes_por_receta)), 2),
            "mediana_ingredientes": int(np.median(ingredientes_por_receta)),
        })

    return pd.DataFrame(reporte)


# Ejecución sobre tu dataframe:
df_comparativa = comparar_cobertura_min_df(df["ingredients"], umbrales=[5, 10], min_ingredientes=3)
df_comparativa

c:\Users\santi\Documents\Codigo\Sistemas recomendadores\tareas\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,min_df,vocabulario,recetas_>=_3,%_recetas_>=_3,recetas_huerfanas_(=0),%_huerfanas,promedio_ingredientes,mediana_ingredientes
0,5,1214,6748,95.76%,9,0.13%,7.81,7
1,10,713,6574,93.29%,40,0.57%,7.34,7


In [15]:
for min_df in [5, 10]:
    vec = CountVectorizer(
        tokenizer=lambda x: x, preprocessor=lambda x: x, lowercase=False, binary=True, min_df=min_df
    )
    conteo = np.asarray(vec.fit_transform(df["ingredients"]).sum(axis=1)).ravel()
    
    print(f"\n--- Distribución con min_df={min_df} (Vocabulario: {len(vec.vocabulary_)}) ---")
    print(f"P10 (10% peor cubierto): {np.percentile(conteo, 10):.0f} ingredientes")
    print(f"P25:                    {np.percentile(conteo, 25):.0f} ingredientes")
    print(f"P50 (Mediana):          {np.percentile(conteo, 50):.0f} ingredientes")
    print(f"P75:                    {np.percentile(conteo, 75):.0f} ingredientes")


--- Distribución con min_df=5 (Vocabulario: 1214) ---
P10 (10% peor cubierto): 4 ingredientes
P25:                    5 ingredientes
P50 (Mediana):          7 ingredientes
P75:                    10 ingredientes

--- Distribución con min_df=10 (Vocabulario: 713) ---
P10 (10% peor cubierto): 3 ingredientes
P25:                    5 ingredientes
P50 (Mediana):          7 ingredientes
P75:                    10 ingredientes


c:\Users\santi\Documents\Codigo\Sistemas recomendadores\tareas\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [16]:
import numpy as np
import torch
from sklearn.feature_extraction.text import CountVectorizer

vec_ingredients = CountVectorizer(
    tokenizer=lambda x: x,
    preprocessor=lambda x: x,
    lowercase=False,
    binary=True,
    min_df=5,
)

# Matriz sparse (7047, 1214)
ing_sparse = vec_ingredients.fit_transform(df["ingredients"])

# Conversión directa a tensor denso para PyTorch
ing_tensor = torch.from_numpy(ing_sparse.toarray().astype(np.float32))

print(f"Tensor de ingredientes listo: {ing_tensor.shape}")

ModuleNotFoundError: No module named 'torch'